In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

# Conexión al Data Warehouse f1_dw
engine_dw = create_engine(f"postgresql+psycopg2://{os.getenv('PG_USER')}:{os.getenv('PG_PASS')}@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/f1_dw")

In [ ]:
# Conteo de registros

query_conteo = """
SELECT 'dim_drivers' AS tabla, COUNT(*) AS total FROM dim_drivers
UNION ALL
SELECT 'dim_constructors', COUNT(*) FROM dim_constructors
UNION ALL
SELECT 'dim_circuits', COUNT(*) FROM dim_circuits
UNION ALL
SELECT 'dim_races_time', COUNT(*) FROM dim_races_time
UNION ALL
SELECT 'dim_status', COUNT(*) FROM dim_status
UNION ALL
SELECT 'fact_results', COUNT(*) FROM fact_results;
"""
df_conteo = pd.read_sql(query_conteo, engine_dw)
display(df_conteo)

,tabla,total
0,dim_drivers,861
1,dim_constructors,212
2,dim_circuits,77
3,dim_races_time,1125
4,dim_status,139
5,fact_results,26759


In [13]:
# Cruces de Datos

query_cruces = """
SELECT 
    d.driver_ref as driver_ref,
    d.full_name as driver_full_name,
    d.nationality as driver_nationality,
    d.dob as date_of_birth,
    c.constructor_ref as constructor_ref,
    c.name as constructor_name,
    c.nationality as constructor_nationality,
    ci.circuit_ref as circuit_ref,
    ci.name as circuit_name,
    ci.location as circuit_location,
    ci.country as circuit_country,
    rc.year as race_year,
    rc.round as race_round,
    rc.name as race_name,
    rc.date as race_date,
    s.status as status,
    f.grid as grid,
    f.position_order as position_order,
    f.points as points,
    f.laps as laps,
    f.milliseconds as milliseconds
FROM fact_results f
JOIN dim_races_time rc ON f.race_id = rc.race_id
JOIN dim_circuits ci ON f.circuit_id = ci.circuit_id
JOIN dim_drivers d ON f.driver_id = d.driver_id
JOIN dim_constructors c ON f.constructor_id = c.constructor_id
JOIN dim_status s ON f.status_id = s.status_id
ORDER BY rc.year DESC, f.position_order ASC
LIMIT 10;
"""
df_cruces = pd.read_sql(query_cruces, engine_dw)
display(df_cruces)

,driver_ref,driver_full_name,driver_nationality,date_of_birth,constructor_ref,constructor_name,constructor_nationality,circuit_ref,circuit_name,circuit_location,...,race_year,race_round,race_name,race_date,status,grid,position_order,points,laps,milliseconds
0,leclerc,Charles Leclerc,Monegasque,1997-10-16,ferrari,Ferrari,Italian,monaco,Circuit de Monaco,Monte-Carlo,...,2024,8,Monaco Grand Prix,2024-05-26,Finished,1,1,25.0,78,8595554
1,max_verstappen,Max Verstappen,Dutch,1997-09-30,red_bull,Red Bull,Austrian,villeneuve,Circuit Gilles Villeneuve,Montreal,...,2024,9,Canadian Grand Prix,2024-06-09,Finished,2,1,25.0,70,6347927
2,norris,Lando Norris,British,1999-11-13,mclaren,McLaren,British,miami,Miami International Autodrome,Miami,...,2024,6,Miami Grand Prix,2024-05-05,Finished,5,1,25.0,57,5449876
3,max_verstappen,Max Verstappen,Dutch,1997-09-30,red_bull,Red Bull,Austrian,imola,Autodromo Enzo e Dino Ferrari,Imola,...,2024,7,Emilia Romagna Grand Prix,2024-05-19,Finished,1,1,25.0,63,5125252
4,max_verstappen,Max Verstappen,Dutch,1997-09-30,red_bull,Red Bull,Austrian,shanghai,Shanghai International Circuit,Shanghai,...,2024,5,Chinese Grand Prix,2024-04-21,Finished,1,1,25.0,56,6052554
5,max_verstappen,Max Verstappen,Dutch,1997-09-30,red_bull,Red Bull,Austrian,jeddah,Jeddah Corniche Circuit,Jeddah,...,2024,2,Saudi Arabian Grand Prix,2024-03-09,Finished,1,1,25.0,50,4843273
6,max_verstappen,Max Verstappen,Dutch,1997-09-30,red_bull,Red Bull,Austrian,bahrain,Bahrain International Circuit,Sakhir,...,2024,1,Bahrain Grand Prix,2024-03-02,Finished,1,1,26.0,57,5504742
7,max_verstappen,Max Verstappen,Dutch,1997-09-30,red_bull,Red Bull,Austrian,suzuka,Suzuka Circuit,Suzuka,...,2024,4,Japanese Grand Prix,2024-04-07,Finished,1,1,26.0,53,6863566
8,sainz,Carlos Sainz,Spanish,1994-09-01,ferrari,Ferrari,Italian,albert_park,Albert Park Grand Prix Circuit,Melbourne,...,2024,3,Australian Grand Prix,2024-03-24,Finished,2,1,25.0,58,4826843
9,max_verstappen,Max Verstappen,Dutch,1997-09-30,red_bull,Red Bull,Austrian,catalunya,Circuit de Barcelona-Catalunya,Montmeló,...,2024,10,Spanish Grand Prix,2024-06-23,Finished,2,1,25.0,66,5300227


In [3]:
# Top Pilotos

query_top = """
SELECT 
    d.full_name,
    d.nationality,
    SUM(f.points) AS total_puntos
FROM fact_results f
JOIN dim_drivers d ON f.driver_id = d.driver_id
GROUP BY d.driver_id, d.full_name, d.nationality
ORDER BY total_puntos DESC
LIMIT 5;
"""
df_top = pd.read_sql(query_top, engine_dw)
display(df_top)

,full_name,nationality,total_puntos
0,Lewis Hamilton,British,4820.5
1,Sebastian Vettel,German,3098.0
2,Max Verstappen,Dutch,2912.5
3,Fernando Alonso,Spanish,2329.0
4,Kimi Räikkönen,Finnish,1873.0
